## Load the Home Credit data from the shared Google Drive
Run the cell below to load the application tables. Call `load_table("bureau")` (or another table name) when you need a history table. Each CSV is downloaded only when first used and cached in your user cache outside this repository. Google Drive remains the shared source; pandas holds each loaded table in memory while the notebook runs.

In [21]:
%pip install -q gdown pandas

import os
from pathlib import Path
import gdown
import pandas as pd
import numpy as np

# File IDs from the shared Google Drive folder.
DRIVE_FILE_IDS = {
    "application_test": "1aSCg7GlC4hluYPybtM00p9FzDWJHHtql",
    "application_train": "1leSsunoq4XyViBVgrZ0dCIEWQePWi_AV",
    "bureau": "1XoTBbU-SMbjz2G8h12TxiZpjJIa-cZh_",
    "bureau_balance": "1vDQPrD5LEc3E4FFBrpLqWaPRq3ZRg4UP",
    "credit_card_balance": "10gP4aVPjxhXt_uWanDNHeUV2HCA_j0Vp",
    "HomeCredit_columns_description": "1z95kRXkg5bPPAenOFPnsE8pZwQcznCok",
    "installments_payments": "19LEAwLvsJCOMuQ-GMmKNeQr8YNA4glVd",
    "POS_CASH_balance": "1_e2CfFcyrxki6j9KI5azFoQIP0EqlmYw",
    "previous_application": "1UltfcT1hOhF3S8e0VKYFyeKR7DGzisOd",
    "sample_submission": "1YNKdy7FL30LaeRLV7r1uyzQY6gOXJBFj",
}

cache_root = Path(os.environ.get("LOCALAPPDATA", str(Path.home() / ".cache")))
DATA_DIR = cache_root / "data4401-project2" / "home-credit-default-risk"

def load_table(name, **read_csv_kwargs):
    """Download a table once, then read it into a pandas DataFrame."""
    if name not in DRIVE_FILE_IDS:
        raise ValueError(f"Unknown table {name!r}. Choose from {sorted(DRIVE_FILE_IDS)}")
    csv_path = DATA_DIR / f"{name}.csv"
    if not csv_path.is_file():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        gdown.download(
            id=DRIVE_FILE_IDS[name], output=str(csv_path),
            quiet=False, use_cookies=False, resume=True,
        )
    return pd.read_csv(csv_path, **read_csv_kwargs)

application_train = load_table("application_train")
application_test = load_table("application_test")
print(f"Loaded train: {application_train.shape}; test: {application_test.shape}")
print(f"Other tables available: {sorted(set(DRIVE_FILE_IDS) - {'application_train', 'application_test'})}")

Note: you may need to restart the kernel to use updated packages.
Loaded train: (307511, 122); test: (48744, 121)
Other tables available: ['HomeCredit_columns_description', 'POS_CASH_balance', 'bureau', 'bureau_balance', 'credit_card_balance', 'installments_payments', 'previous_application', 'sample_submission']


# Pre-Processing

In [22]:
application_train = load_table("application_train")

application_train.describe(include="all").T[["count", "mean", "std", "min", "max"]].sort_values("count")

print(application_train.columns)

Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='object', length=122)


In [23]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
application_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


In [25]:
# application_train["TARGET"].value_counts()
application_train["TARGET"].value_counts(normalize=True) * 100

TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

In [26]:
missing = (
    application_train.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing.head(30)

COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MODE       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
FLOORSMIN_AVG               67.848630
FLOORSMIN_MODE              67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_MODE               59.376738
LANDAREA_AVG                59.376738
BASEMENTAREA_MEDI           58.515956
BASEMENTAREA_AVG            58.515956
BASEMENTAREA_MODE           58.515956
EXT_SOURCE_1                56.381073
NONLIVINGAREA_MODE          55.179164
NONLIVINGAREA_AVG           55.179164
NONLIVINGARE

In [27]:
numeric_cols = application_train.select_dtypes(include=np.number).columns
categorical_cols = application_train.select_dtypes(exclude=np.number).columns

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols.tolist())

Numeric: 106
Categorical: 16

Categorical columns:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [28]:
application_train.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
TARGET,307511.0,0.080729,0.272419,0.0,0.0,0.0,0.0,1.0
CNT_CHILDREN,307511.0,0.417052,0.722121,0.0,0.0,0.0,1.0,19.0
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,265992.0,0.007000,0.110757,0.0,0.0,0.0,0.0,9.0
AMT_REQ_CREDIT_BUREAU_WEEK,265992.0,0.034362,0.204685,0.0,0.0,0.0,0.0,8.0
AMT_REQ_CREDIT_BUREAU_MON,265992.0,0.267395,0.916002,0.0,0.0,0.0,0.0,27.0
AMT_REQ_CREDIT_BUREAU_QRT,265992.0,0.265474,0.794056,0.0,0.0,0.0,0.0,261.0


In [29]:
# Target distribution
print("TARGET distribution:")
print(application_train["TARGET"].value_counts())
print()
print(application_train["TARGET"].value_counts(normalize=True) * 100)

# Missing values
missing = application_train.isnull().mean().mul(100).sort_values(ascending=False)

print("\nColumns with missing values:")
print(missing[missing > 0])

TARGET distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

Columns with missing values:
COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
                              ...    
EXT_SOURCE_2                 0.214626
AMT_GOODS_PRICE              0.090403
AMT_ANNUITY                  0.003902
CNT_FAM_MEMBERS              0.000650
DAYS_LAST_PHONE_CHANGE       0.000325
Length: 67, dtype: float64


In [30]:
missing_table = pd.DataFrame({
    "missing_count": application_train.isnull().sum(),
    "missing_percent": application_train.isnull().mean() * 100
})

missing_table = (
    missing_table[missing_table["missing_count"] > 0]
    .sort_values("missing_percent", ascending=False)
)

missing_table.head(20)

,missing_count,missing_percent
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_AVG,214865,69.872297
COMMONAREA_MODE,214865,69.872297
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_MODE,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953
LIVINGAPARTMENTS_AVG,210199,68.354953


In [31]:
print("Over 60% missing:",
      (missing_table["missing_percent"] > 60).sum())

print("40-60% missing:",
      ((missing_table["missing_percent"] > 40) &
       (missing_table["missing_percent"] <= 60)).sum())

print("20-40% missing:",
      ((missing_table["missing_percent"] > 20) &
       (missing_table["missing_percent"] <= 40)).sum())

print("Under 20% missing:",
      (missing_table["missing_percent"] <= 20).sum())

Over 60% missing: 17
40-60% missing: 32
20-40% missing: 1
Under 20% missing: 17


In [33]:
ids = application_train["SK_ID_CURR"].copy()

y = application_train["TARGET"].copy()

X = application_train.drop(
    columns=["TARGET", "SK_ID_CURR"]
).copy()

# 1. Random Split

In [35]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train_random, X_temp_random, y_train_random, y_temp_random = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Split the temporary 30% equally:
# 15% validation and 15% test
X_val_random, X_test_random, y_val_random, y_test_random = train_test_split(
    X_temp_random,
    y_temp_random,
    test_size=0.50,
    random_state=42
)

print("Training:", X_train_random.shape)
print("Validation:", X_val_random.shape)
print("Test:", X_test_random.shape)

Training: (215257, 120)
Validation: (46127, 120)
Test: (46127, 120)


# 2. Stratified Split

In [36]:
# STRATIFIED SPLIT

# First split: 70% training, 30% temporary
X_train_strat, X_temp_strat, y_train_strat, y_temp_strat = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split temporary set into 15% validation and 15% test
X_val_strat, X_test_strat, y_val_strat, y_test_strat = train_test_split(
    X_temp_strat,
    y_temp_strat,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_strat
)

print("Training:", X_train_strat.shape)
print("Validation:", X_val_strat.shape)
print("Test:", X_test_strat.shape)

Training: (215257, 120)
Validation: (46127, 120)
Test: (46127, 120)


In [37]:
print("Original:")
print(y.value_counts(normalize=True) * 100)

print("\nTraining:")
print(y_train_strat.value_counts(normalize=True) * 100)

print("\nValidation:")
print(y_val_strat.value_counts(normalize=True) * 100)

print("\nTest:")
print(y_test_strat.value_counts(normalize=True) * 100)

Original:
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

Training:
TARGET
0    91.927324
1     8.072676
Name: proportion, dtype: float64

Validation:
TARGET
0    91.926637
1     8.073363
Name: proportion, dtype: float64

Test:
TARGET
0    91.926637
1     8.073363
Name: proportion, dtype: float64
